# Mobile Money Agent Network: Audit, Cleaning and Summary

**Engagement 2 · Prepared for:** the Regional Manager and the Data Governance team
**Question:** which agents and counties handle the most deposits and withdrawals, once the unreliable legacy extract has been audited and cleaned?
**Inputs:** `agent_transactions.csv` and `agents.csv` (seed 44, produced by `generate_data.py`) · **Pipeline:** `cleaning_pipeline.py` → `clean_agent_data(txns, agents)` · **Currency:** KES

## 0. Setup and loading the raw files
Everything is loaded exactly as delivered. I also load a second, **all-text** copy (`dtype=str, keep_default_na=False`) for the audit: by default pandas silently turns the text `"N/A"` into a missing value, which would make blanks and `"N/A"` strings indistinguishable.

In [1]:
import subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd

from cleaning_pipeline import clean_agent_data

if not Path("agent_transactions.csv").exists() or not Path("agents.csv").exists():
    subprocess.run([sys.executable, "generate_data.py"], check=True)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

txns_raw = pd.read_csv("agent_transactions.csv")
agents = pd.read_csv("agents.csv")
txns_text = pd.read_csv("agent_transactions.csv", dtype=str, keep_default_na=False)   # audit copy: nothing hidden by pandas defaults

print("agent_transactions.csv:", txns_raw.shape, "| agents.csv:", agents.shape)

agent_transactions.csv: (612, 5) | agents.csv: (20, 3)


## 1. First look at the raw data

In [2]:
display(txns_raw.head(8))
display(agents.head(5))
txns_raw.info()

,txn_id,agent_id,txn_type,amount,txn_date
0,TX00001,AG009,deposit,353.0,2026-03-03
1,TX00002,AG020,deposit,1238.0,2026-04-27
2,TX00003,AG008,DEPOSIT,9129.0,2026-04-29
3,TX00004,AG009,deposit,652.0,2026-04-24
4,TX00005,AG009,withdrawal,478.0,2026-04-17
5,TX00006,AG019,withdrawal,931.0,2026-04-23
6,TX00007,AG020,deposit,2583.0,2026-04-02
7,TX00008,AG019,deposit,1797.0,2026-04-12


,agent_id,county,onboarded
0,AG001,Mombasa,2024-01-01
1,AG002,Nairobi,2024-02-01
2,AG003,Kisumu,2024-03-01
3,AG004,Kiambu,2024-04-01
4,AG005,Mombasa,2024-05-01


<class 'pandas.DataFrame'>
RangeIndex: 612 entries, 0 to 611
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   txn_id    612 non-null    str    
 1   agent_id  612 non-null    str    
 2   txn_type  612 non-null    str    
 3   amount    590 non-null    float64
 4   txn_date  612 non-null    str    
dtypes: float64(1), str(4)
memory usage: 24.0 KB


In [3]:
print("Missing values per column (default pandas load):")
print(txns_raw.isna().sum().to_string())
print("\ntxn_type values exactly as stored:")
print(txns_raw["txn_type"].map(repr).value_counts().to_string())
print("\nagent_ids in transactions but not in the register:", sorted(set(txns_raw["agent_id"]) - set(agents["agent_id"])))
print("Rows whose txn_date contains '/':", txns_raw["txn_date"].str.contains("/").sum())
print("Duplicated txn_id values (surplus rows):", txns_raw["txn_id"].duplicated().sum(),
      "| exact full-row duplicates:", txns_raw.duplicated().sum())

Missing values per column (default pandas load):
txn_id       0
agent_id     0
txn_type     0
amount      22
txn_date     0

txn_type values exactly as stored:
txn_type
'deposit'       142
'withdrawal'    131
'Deposit'       128
'DEPOSIT '      111
'Withdrawal'    100

agent_ids in transactions but not in the register: ['AG099']
Rows whose txn_date contains '/': 5
Duplicated txn_id values (surplus rows): 12 | exact full-row duplicates: 11


## 2. Data-quality audit table
One row per issue, classified by the six dimensions of data quality. Counts are **raw rows in the 612-row extract** (before de-duplication). Each count is computed in code, none is typed in by hand.

In [4]:
n_raw = len(txns_raw)
amt = txns_text["amount"].str.strip()
is_blank = amt.eq("")
is_na_text = amt.str.upper().eq("N/A")
amt_num = pd.to_numeric(amt.mask(is_blank | is_na_text), errors="coerce")
other_invalid = int((amt_num.isna() & ~is_blank & ~is_na_text).sum())            # strings that are neither blank, N/A nor numeric

canon = {"Deposit", "Withdrawal"}
noncanon = ~txns_text["txn_type"].isin(canon)
padded = txns_text["txn_type"].ne(txns_text["txn_type"].str.strip())

dup_surplus = int(txns_raw["txn_id"].duplicated().sum())
exact_surplus = int(txns_raw.duplicated().sum())
unknown_raw = ~txns_raw["agent_id"].isin(agents["agent_id"])

first_copy = txns_raw.drop_duplicates("txn_id")                                  # IQR on one copy of each transaction
a = pd.to_numeric(first_copy["amount"], errors="coerce").dropna()
q1, q3 = a.quantile([0.25, 0.75]); iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
n_iqr = int(((a < lo) | (a > hi)).sum())
n_max_value = int((txns_raw["amount"] == a.max()).sum())

dates = pd.to_datetime(txns_text["txn_date"].str.replace("/", "-"), errors="coerce")
onb = pd.to_datetime(txns_raw["agent_id"].map(agents.set_index("agent_id")["onboarded"]))
n_timeliness = int((dates > pd.Timestamp.today()).sum() + (dates < onb).sum())

audit = pd.DataFrame([
    ("Same txn_id appears more than once (surplus copies)",                        "Uniqueness",   "txn_id",              dup_surplus),
    ("...of which the copy differs from the original (agent_id overwritten)",       "Uniqueness",   "txn_id, agent_id",    dup_surplus - exact_surplus),
    ("txn_type not exactly 'Deposit'/'Withdrawal' (mixed case)",                    "Consistency",  "txn_type",            int(noncanon.sum())),
    ("...of which also carry leading/trailing spaces (e.g. 'DEPOSIT ')",            "Consistency",  "txn_type",            int(padded.sum())),
    ("agent_id not in the agent register (AG099)",                                  "Consistency",  "agent_id",            int(unknown_raw.sum())),
    ("amount is blank / missing",                                                   "Completeness", "amount",              int(is_blank.sum())),
    ("amount is the text 'N/A' (pandas reads it as NaN by default)",                "Validity",     "amount",              int(is_na_text.sum())),
    ("amount is another non-numeric string",                                        "Validity",     "amount",              other_invalid),
    ("txn_date written YYYY/MM/DD instead of YYYY-MM-DD",                           "Validity",     "txn_date",            int(txns_text["txn_date"].str.contains("/").sum())),
    ("txn_date stored as text, not a datetime",                                     "Validity",     "txn_date",            n_raw),
    ("onboarded stored as text, not a datetime (agents.csv)",                       "Validity",     "onboarded",           len(agents)),
    (f"Same extreme amount repeated ({a.max():,.0f} = {a.max()/a.median():.0f}x the median)", "Accuracy", "amount",         n_max_value),
    ("amount outside the IQR fences (includes the extreme values above)",           "Accuracy",     "amount",              n_iqr),
    ("Dates in the future or before the agent was onboarded",                       "Timeliness",   "txn_date",            n_timeliness),
], columns=["issue", "dimension", "column", "rows_affected"])
audit["pct_of_rows"] = (audit["rows_affected"] / n_raw * 100).round(1)
dim_order = ["Completeness", "Validity", "Accuracy", "Consistency", "Uniqueness", "Timeliness"]
audit = audit.sort_values("dimension", key=lambda s: s.map(dim_order.index), kind="stable").reset_index(drop=True)
pd.set_option("display.max_colwidth", 90)
audit

,issue,dimension,column,rows_affected,pct_of_rows
0,amount is blank / missing,Completeness,amount,15,2.5
1,amount is the text 'N/A' (pandas reads it as NaN by default),Validity,amount,7,1.1
2,amount is another non-numeric string,Validity,amount,0,0.0
3,txn_date written YYYY/MM/DD instead of YYYY-MM-DD,Validity,txn_date,5,0.8
4,"txn_date stored as text, not a datetime",Validity,txn_date,612,100.0
5,"onboarded stored as text, not a datetime (agents.csv)",Validity,onboarded,20,3.3
6,"Same extreme amount repeated (250,000 = 141x the median)",Accuracy,amount,5,0.8
7,amount outside the IQR fences (includes the extreme values above),Accuracy,amount,40,6.5
8,txn_type not exactly 'Deposit'/'Withdrawal' (mixed case),Consistency,txn_type,384,62.7
9,...of which also carry leading/trailing spaces (e.g. 'DEPOSIT '),Consistency,txn_type,111,18.1


In [5]:
print("Rows affected by dimension (an issue can hit several rows or several dimensions at once):")
print(audit.groupby("dimension")["rows_affected"].agg(issues="count", max_rows_affected="max").reindex(dim_order).to_string())

Rows affected by dimension (an issue can hit several rows or several dimensions at once):
              issues  max_rows_affected
dimension                              
Completeness       1                 15
Validity           5                612
Accuracy           2                 40
Consistency        3                384
Uniqueness         2                 12
Timeliness         1                  0


### Reading the audit
- **Completeness:** 15 blank amounts, plus 7 `"N/A"` strings that pandas reads as `NaN`: **22 rows** are missing under `isna()`, but only 15 would be found by looking for blanks alone.
- **Uniqueness:** 12 surplus copies, but only 11 are exact full-row copies. One copy had its `agent_id` overwritten with `AG099`, so a plain `drop_duplicates()` would leave a duplicate `txn_id` behind. The pipeline therefore de-duplicates on `txn_id`.
- **Hidden in plain sight:** `txn_date` looks fine until you notice 5 rows use `/`; and sorting dates as text puts `2026/04/28` after `2026-04-29`.
- **Accuracy:** the IQR rule flags 40 amounts, but 5 of them are the *same* value, 250,000, which is far from the rest (section 5 below).

## 3. Cleaning pipeline: one call, a log, and proof that it worked
All rules live in `cleaning_pipeline.py::clean_agent_data(txns, agents)`; no row numbers are hard-coded and nothing is edited by hand. The notebook only imports and calls it.

In [6]:
clean, log = clean_agent_data(txns_raw, agents)
for line in log:
    print("-", line)

- Input: 612 transaction rows, 20 agents in register
- Consistency: standardised txn_type on 384 rows to ['Deposit', 'Withdrawal']; 0 unmappable values left missing
- Validity: amount converted to numeric; 22 missing (blank or 'N/A'), 0 non-numeric strings, 0 non-positive values -> set to missing
- Validity: txn_date converted to datetime; 5 rows used the YYYY/MM/DD format and were normalised; 0 unparseable dates left as NaT
- Uniqueness: removed 12 duplicate transactions (one row kept per txn_id); 11 were exact full-row copies and 1 differed in at least one field, so drop_duplicates() alone would have missed them
- Accuracy: IQR rule on 580 genuine amounts (Q1=1,077, Q3=3,080, fences -1,928 to 6,084); flagged 40 outliers (kept, not deleted)
- Completeness: flagged and imputed 20 missing/invalid amounts with the median non-outlier amount of the same txn_type {'Deposit': 1717.0, 'Withdrawal': 1646.0}
- Consistency: left-joined to agent register (600 rows before and after); 7 transaction

In [7]:
before_after = pd.Series({
    "Raw transaction rows": len(txns_raw),
    "Duplicate transactions removed": len(txns_raw) - len(clean),
    "Clean transaction rows": len(clean),
    "Distinct txn_id in raw data": txns_raw["txn_id"].nunique(),
})
print(before_after.to_string())

# ---- validation of every rule -------------------------------------------------
assert len(clean) == txns_raw["txn_id"].nunique()                      # one row per transaction, nothing else lost
assert clean["txn_id"].is_unique                                       # Uniqueness
assert set(clean["txn_type"]) == {"Deposit", "Withdrawal"}             # Consistency
assert pd.api.types.is_float_dtype(clean["amount"]) and clean["amount"].notna().all()      # Validity / Completeness
assert pd.api.types.is_datetime64_any_dtype(clean["txn_date"]) and clean["txn_date"].notna().all()
assert clean["county"].notna().all()
assert clean_agent_data(txns_raw, agents)[0].equals(clean)             # re-running gives identical output (deterministic)
print("\nAll validation checks passed.")
clean.dtypes.to_frame("dtype").T

Raw transaction rows              612
Duplicate transactions removed     12
Clean transaction rows            600
Distinct txn_id in raw data       600

All validation checks passed.


,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier,county,unknown_agent,onboarded
dtype,string,string,str,float64,datetime64[us],bool,bool,string,bool,datetime64[us]


### Decision: duplicates
A duplicate is a repeated `txn_id`, not a repeated row. Exactly one of the 12 duplicates differs from its original: `TX00405` appears once with agent `AG014` and once with `AG099`. The pipeline keeps the copy whose agent is in the register (and, failing that, the one with an amount), so the valid agent is not thrown away.

In [8]:
txns_raw[txns_raw["txn_id"] == "TX00405"]

,txn_id,agent_id,txn_type,amount,txn_date
404,TX00405,AG014,Deposit,1936.0,2026-04-03
601,TX00405,AG099,Deposit,1936.0,2026-04-03


## 4. Completeness: missing and invalid amounts

In [9]:
imputed = clean[clean["amount_imputed"]]
genuine = clean.loc[~clean["amount_imputed"], "amount"]

print(f"Transactions with missing/invalid amount: {len(imputed)} of {len(clean)} ({len(imputed)/len(clean)*100:.1f}%)")
print(f"Value created by imputation: KES {imputed['amount'].sum():,.0f} = {imputed['amount'].sum()/clean['amount'].sum()*100:.1f}% of total value "
      f"({imputed['amount'].sum()/genuine.sum()*100:.1f}% of genuine value)")
print("\nMedian used for each type (non-outlier genuine amounts):")
print(clean[~clean["amount_imputed"] & ~clean["is_outlier"]].groupby("txn_type")["amount"].median().to_string())
print("\nImputed rows by type:", imputed["txn_type"].value_counts().to_dict())

# Sensitivity: does imputation change which counties come first?
rank_all = clean.groupby("county")["amount"].sum().rank(ascending=False).astype(int)
rank_genuine = clean[~clean["amount_imputed"]].groupby("county")["amount"].sum().rank(ascending=False).astype(int)
display(pd.DataFrame({"rank_with_imputed": rank_all, "rank_genuine_only": rank_genuine}).sort_values("rank_with_imputed"))
imputed.head(6)

Transactions with missing/invalid amount: 20 of 600 (3.3%)
Value created by imputation: KES 33,843 = 1.3% of total value (1.3% of genuine value)

Median used for each type (non-outlier genuine amounts):
txn_type
Deposit       1717.0
Withdrawal    1646.0

Imputed rows by type: {'Deposit': 13, 'Withdrawal': 7}


,rank_with_imputed,rank_genuine_only
county,,
Nairobi,1,1
Mombasa,2,2
Nakuru,3,3
Kisumu,4,4
Kiambu,5,5
Unknown,6,6


,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier,county,unknown_agent,onboarded
10,TX00011,AG009,Withdrawal,1646.0,2026-03-27,True,False,Nakuru,False,2024-09-01
14,TX00015,AG003,Deposit,1717.0,2026-03-18,True,False,Kisumu,False,2024-03-01
54,TX00055,AG012,Withdrawal,1646.0,2026-03-19,True,False,Kisumu,False,2024-12-01
66,TX00067,AG002,Deposit,1717.0,2026-04-07,True,False,Nairobi,False,2024-02-01
122,TX00123,AG008,Deposit,1717.0,2026-04-15,True,False,Kisumu,False,2024-08-01
140,TX00141,AG004,Withdrawal,1646.0,2026-03-09,True,False,Kiambu,False,2024-04-01


### Decision and justification
**What I did:** *flag first, then fill.* `amount_imputed = True` marks the 20 affected transactions (3.3%), and their `amount` is filled with the **median of the genuine, non-outlier amounts of the same transaction type** (Deposit KES 1,717, Withdrawal KES 1,646).

**Why not the alternatives**
- *Drop the rows:* these are real transactions that happened (they have an ID, agent, type and date). Deleting them would understate every agent's transaction count and break reconciliation with the legacy system, for the sake of a missing value.
- *Fill with the mean:* amounts are strongly right-skewed and contain extreme values, so the mean (KES 4,574 with the extremes) is not a typical transaction; the median is robust to them. Using the median *by type* respects that deposits and withdrawals are different products.
- *Leave as missing:* the totals would silently exclude those transactions while counts include them.

**What it costs:** imputation adds KES 33,843, 1.3% of total value, and does not change the ranking of counties by value (table above). Because the flag is kept, any figure can be recomputed on genuine amounts only, and the 20 real values should be recovered from the source ledger. Imputed values must never be used to judge an individual agent.

## 5. Outliers: IQR method and the KES 250,000 values
The IQR fences are computed on **genuine** amounts (after de-duplication, before imputation), so repeated or invented values cannot move them.

In [10]:
q1, q3 = genuine.quantile([0.25, 0.75]); iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"Q1 = {q1:,.0f}   Q3 = {q3:,.0f}   IQR = {iqr:,.0f}   fences = [{lower:,.0f}, {upper:,.0f}]")
print(f"Flagged: {clean['is_outlier'].sum()} of {len(genuine)} genuine amounts ({clean['is_outlier'].sum()/len(genuine)*100:.1f}%)")
assert clean["is_outlier"].sum() == ((genuine < lower) | (genuine > upper)).sum() == n_iqr

# How extreme is each flagged value? Robust z-score on the log scale (amounts are log-normal-ish)
logs = np.log(genuine)
robust_sd = 1.4826 * (logs - logs.median()).abs().median()
flagged = clean[clean["is_outlier"]].copy()
flagged["x_median"] = (flagged["amount"] / genuine.median()).round(1)
flagged["log_z"] = ((np.log(flagged["amount"]) - logs.median()) / robust_sd).round(1)
display(flagged.sort_values("amount", ascending=False).head(10)[["txn_id", "agent_id", "county", "txn_type", "amount", "txn_date", "x_median", "log_z"]])

extreme = flagged["log_z"] > 5
suspect_ids = flagged.loc[extreme, "txn_id"]
suspect = clean["txn_id"].isin(suspect_ids)
natural = flagged[~extreme]
print(f"\nFlagged but plausible (heavy right tail): {len(natural)} values, KES {natural['amount'].min():,.0f} to {natural['amount'].max():,.0f}, max log_z = {natural['log_z'].max()}")
print(f"Extreme: {suspect.sum()} values, all equal to {clean.loc[suspect, 'amount'].unique()}, log_z = {flagged.loc[extreme, 'log_z'].unique()}")
print(f"They are {suspect.sum()/len(clean)*100:.2f}% of transactions but {clean.loc[suspect, 'amount'].sum()/clean['amount'].sum()*100:.1f}% of total value.")
print("\nSpread across:", "counties =", clean.loc[suspect, "county"].tolist(), "| types =", clean.loc[suspect, "txn_type"].value_counts().to_dict())

Q1 = 1,077   Q3 = 3,080   IQR = 2,003   fences = [-1,928, 6,084]
Flagged: 40 of 580 genuine amounts (6.9%)


,txn_id,agent_id,county,txn_type,amount,txn_date,x_median,log_z
424,TX00425,AG005,Mombasa,Withdrawal,250000.0,2026-03-02,140.7,6.6
463,TX00464,AG009,Nakuru,Withdrawal,250000.0,2026-04-11,140.7,6.6
315,TX00316,AG014,Nairobi,Deposit,250000.0,2026-04-01,140.7,6.6
594,TX00595,AG016,Kiambu,Deposit,250000.0,2026-03-11,140.7,6.6
306,TX00307,AG012,Kisumu,Deposit,250000.0,2026-03-02,140.7,6.6
183,TX00184,AG006,Nakuru,Deposit,28847.0,2026-03-08,16.2,3.7
222,TX00223,AG011,Kisumu,Deposit,22316.0,2026-03-02,12.6,3.4
573,TX00574,AG017,Nakuru,Deposit,16605.0,2026-03-16,9.3,3.0
174,TX00175,AG014,Nairobi,Withdrawal,13362.0,2026-04-06,7.5,2.7
212,TX00213,AG019,Mombasa,Deposit,12078.0,2026-04-10,6.8,2.6



Flagged but plausible (heavy right tail): 35 values, KES 6,158 to 28,847, max log_z = 3.7
Extreme: 5 values, all equal to [250000.], log_z = [6.6]
They are 0.83% of transactions but 46.5% of total value.

Spread across: counties = ['Kisumu', 'Nairobi', 'Mombasa', 'Nakuru', 'Kiambu'] | types = {'Deposit': 3, 'Withdrawal': 2}


### Investigation and recommendation
**The IQR rule flags 40 amounts above KES 6,085, and they are two different things.**
- **35 are an ordinary heavy right tail** (KES 6,158–KES 28,847, at most 3.7 robust standard deviations from the median on the log scale). Large deposits and withdrawals are normal for an agent network, so these stay in the data, flagged `is_outlier=True`, and are *included* in totals.
- **5 are the same value, KES 250,000**, which is 141× the median, 8.7× the next-largest value (KES 28,847) and 6.6 robust standard deviations from the median, where nothing else exceeds 3.7. Identical figures that are this far out are not what organic transactions look like. They also fall on 5 different counties and both transaction types (one per county), so they are not one agent's quirk. Together they are 0.8% of transactions but **46.5% of total value**.

**What they might be:** a placeholder or default value written by the legacy system when an amount fails to capture, a test/ceiling entry (KES 250,000 matches the commonly published single-transaction limit on Kenyan mobile-money platforms, so the provider should confirm its own limit), or genuine maximum-size transactions. The data alone cannot decide between these.

**Recommendation for the business**
1. **Do not delete them and do not "correct" them**: they may be real money movements. They stay in `agent_transactions_clean.csv` with `is_outlier=True`.
2. **Quarantine from decision metrics until verified**: report county and agent values with and without them (done in section 7), and exclude them from *averages*, where these 5 values alone inflate the apparent average transaction by 85%.
3. **Verify against the source ledger / agent float records**, and ask the system owner whether `KES 250,000` is a default or a limit value. Reinstate them in totals if they are confirmed genuine, or correct them if not.
4. **Add a validation rule at source** so that amounts at the system ceiling require a second approval or a reason code.

## 6. Merge validation: transactions → agent register
The join is a **left join validated as many-to-one** (many transactions per agent, one register row per agent), so no transaction can be dropped and none can be multiplied. A transaction whose agent is not in the register is kept with `county = "Unknown"` and `unknown_agent = True`, because it is still a real, valid transaction.

In [11]:
deduped_rows = txns_raw["txn_id"].nunique()
validation = pd.Series({
    "Raw transaction rows": len(txns_raw),
    "Rows after de-duplication (before merge)": deduped_rows,
    "Rows after merge (clean)": len(clean),
    "Agents in register": len(agents),
    "Registered agents with >=1 transaction": clean.loc[~clean["unknown_agent"], "agent_id"].nunique(),
    "Transactions with unknown agent": int(clean["unknown_agent"].sum()),
    "Raw rows with unknown agent (before de-dup)": int(unknown_raw.sum()),
})
print(validation.to_string())
assert len(clean) == deduped_rows, "merge changed the number of rows"

unknown = clean[clean["unknown_agent"]]
print("\nUnknown agent_id(s):", unknown["agent_id"].unique().tolist(),
      f"| value KES {unknown['amount'].sum():,.0f} = {unknown['amount'].sum()/clean['amount'].sum()*100:.1f}% of total value")
print("Transactions dated before the agent's onboarding date:", int((clean["txn_date"] < clean["onboarded"]).sum()))
unknown[["txn_id", "agent_id", "txn_type", "amount", "txn_date", "county", "unknown_agent"]]

Raw transaction rows                           612
Rows after de-duplication (before merge)       600
Rows after merge (clean)                       600
Agents in register                              20
Registered agents with >=1 transaction          20
Transactions with unknown agent                  7
Raw rows with unknown agent (before de-dup)      8

Unknown agent_id(s): ['AG099'] | value KES 27,156 = 1.0% of total value
Transactions dated before the agent's onboarding date: 0


,txn_id,agent_id,txn_type,amount,txn_date,county,unknown_agent
55,TX00056,AG099,Deposit,8507.0,2026-04-01,Unknown,True
68,TX00069,AG099,Withdrawal,1877.0,2026-04-24,Unknown,True
77,TX00078,AG099,Deposit,1770.0,2026-04-06,Unknown,True
237,TX00238,AG099,Deposit,4944.0,2026-04-06,Unknown,True
267,TX00268,AG099,Deposit,3110.0,2026-04-09,Unknown,True
444,TX00445,AG099,Deposit,5309.0,2026-04-27,Unknown,True
557,TX00558,AG099,Deposit,1639.0,2026-03-22,Unknown,True


### Result and treatment
- **Row counts reconcile:** 600 rows before the merge and 600 after, so nothing was lost or duplicated.
- **7 transactions reference an unknown agent** (`AG099`, not in the 20-agent register), worth KES 27,156 or 1.0% of total value. The raw extract had 8; one was a corrupted duplicate of a valid transaction (`TX00405`) and was removed in the de-duplication step, so the genuine problem is 7 rows.
- **Treatment:** kept (dropping them would delete valid money movements), labelled `county = "Unknown"`, flagged `unknown_agent = True`, and shown as their own row in every summary so that totals still reconcile. They cannot be attributed to a county until the data owner confirms whether `AG099` is an unregistered agent or a corrupted ID.

## 7. Summary tables
**(a)** County-level total value, number of transactions and average value, split by transaction type. **(b)** Weekly transaction value by county. Both are on **all clean transactions** (so they reconcile to the clean file), followed by the same view with the five quarantined extreme values excluded, since they distort the picture (section 5).

In [12]:
def summarise(df):
    out = (df.groupby(["county", "txn_type"])
             .agg(total_value=("amount", "sum"), n_transactions=("txn_id", "count"), avg_value=("amount", "mean"))
             .round(0).astype({"total_value": "int64", "avg_value": "int64"}))
    return out

county_summary = summarise(clean)
print("(a) County summary by transaction type: all clean transactions")
display(county_summary)

county_totals = (clean.groupby("county")
                 .agg(total_value=("amount", "sum"), n_transactions=("txn_id", "count"), avg_value=("amount", "mean"))
                 .round(0).sort_values("total_value", ascending=False))
county_totals.loc["ALL COUNTIES"] = [clean["amount"].sum(), len(clean), round(clean["amount"].mean())]
county_totals = county_totals.astype("int64")
display(county_totals)
assert county_summary["total_value"].sum() == clean["amount"].sum() and county_summary["n_transactions"].sum() == len(clean)

(a) County summary by transaction type: all clean transactions


total_value  n_transactions  avg_value
county  txn_type                                          
Kiambu  Deposit          334737              43       7785
        Withdrawal        76920              27       2849
Kisumu  Deposit          436242              74       5895
        Withdrawal        76244              37       2061
Mombasa Deposit          148748              72       2066
        Withdrawal       368592              61       6042
Nairobi Deposit          543352             118       4605
        Withdrawal       161514              62       2605
Nakuru  Deposit          184771              60       3080
        Withdrawal       328464              39       8422
Unknown Deposit           25279               6       4213
        Withdrawal         1877               1       1877

,total_value,n_transactions,avg_value
county,,,
Nairobi,704866,180,3916
Mombasa,517340,133,3890
Nakuru,513235,99,5184
Kisumu,512486,111,4617
Kiambu,411657,70,5881
Unknown,27156,7,3879
ALL COUNTIES,2686740,600,4478


In [13]:
print("(a2) Same view with the 5 quarantined extreme values excluded")
ex = clean[~suspect]
county_totals_ex = (ex.groupby("county")
                    .agg(total_value=("amount", "sum"), n_transactions=("txn_id", "count"), avg_value=("amount", "mean"))
                    .round(0).astype("int64").sort_values("total_value", ascending=False))
display(county_totals_ex)

ranks = pd.DataFrame({
    "rank_by_transactions": county_totals.drop("ALL COUNTIES")["n_transactions"].rank(ascending=False, method="min").astype(int),
    "rank_by_value_all": county_totals.drop("ALL COUNTIES")["total_value"].rank(ascending=False, method="min").astype(int),
    "rank_by_value_excl_quarantined": county_totals_ex["total_value"].rank(ascending=False, method="min").astype(int),
}).sort_values("rank_by_transactions")
display(ranks)

print("Deposits vs withdrawals (all clean transactions, excluding quarantined values in the second table):")
display(clean.groupby("txn_type").agg(total_value=("amount", "sum"), n_transactions=("txn_id", "count"), avg_value=("amount", "mean")).round(0).astype("int64"))
display(ex.groupby("txn_type").agg(total_value=("amount", "sum"), n_transactions=("txn_id", "count"), avg_value=("amount", "mean")).round(0).astype("int64"))

(a2) Same view with the 5 quarantined extreme values excluded


,total_value,n_transactions,avg_value
county,,,
Nairobi,454866,179,2541
Mombasa,267340,132,2025
Nakuru,263235,98,2686
Kisumu,262486,110,2386
Kiambu,161657,69,2343
Unknown,27156,7,3879


,rank_by_transactions,rank_by_value_all,rank_by_value_excl_quarantined
county,,,
Nairobi,1,1,1
Mombasa,2,2,2
Kisumu,3,4,4
Nakuru,4,3,3
Kiambu,5,5,5
Unknown,6,6,6


Deposits vs withdrawals (all clean transactions, excluding quarantined values in the second table):


,total_value,n_transactions,avg_value
txn_type,,,
Deposit,1673129,373,4486
Withdrawal,1013611,227,4465


,total_value,n_transactions,avg_value
txn_type,,,
Deposit,923129,370,2495
Withdrawal,513611,225,2283


In [14]:
# Top agents by value (quarantined values excluded), with the deposit/withdrawal split
agent_view = (ex[~ex["unknown_agent"]]
              .pivot_table(index=["agent_id", "county"], columns="txn_type", values="amount",
                           aggfunc=["sum", "count"], fill_value=0))
agent_view.columns = [f"{'value' if a == 'sum' else 'n'}_{t}" for a, t in agent_view.columns]
agent_view["total_value"] = agent_view["value_Deposit"] + agent_view["value_Withdrawal"]
agent_view["n_transactions"] = agent_view["n_Deposit"] + agent_view["n_Withdrawal"]
top_agents = agent_view.sort_values("total_value", ascending=False).head(5)
print("Top 5 agents by value (quarantined values excluded):")
display(top_agents)
print("Top 5 agents by number of transactions:")
display(agent_view.sort_values("n_transactions", ascending=False).head(5)[["n_Deposit", "n_Withdrawal", "n_transactions", "total_value"]])

Top 5 agents by value (quarantined values excluded):


,,value_Deposit,value_Withdrawal,n_Deposit,n_Withdrawal,total_value,n_transactions
agent_id,county,,,,,,
AG006,Nakuru,75651.0,34307.0,23,14,109958.0,37
AG007,Nairobi,71155.0,35418.0,25,10,106573.0,35
AG017,Nakuru,84693.0,14103.0,26,11,98796.0,37
AG014,Nairobi,55710.0,31879.0,21,10,87589.0,31
AG016,Kiambu,43758.0,41416.0,19,13,85174.0,32


Top 5 agents by number of transactions:


,,n_Deposit,n_Withdrawal,n_transactions,total_value
agent_id,county,,,,
AG004,Kiambu,23,14,37,76483.0
AG006,Nakuru,23,14,37,109958.0
AG017,Nakuru,26,11,37,98796.0
AG007,Nairobi,25,10,35,106573.0
AG003,Kisumu,22,12,34,62153.0


In [15]:
# (b) Weekly pivot: weeks start on Monday; first/last weeks are partial because the extract starts on Sunday 1 Mar and ends Wed 29 Apr
weekly = clean.assign(week_starting=(clean["txn_date"] - pd.to_timedelta(clean["txn_date"].dt.dayofweek, unit="D")).dt.strftime("%Y-%m-%d"))
weekly_pivot = weekly.pivot_table(index="week_starting", columns="county", values="amount",
                                  aggfunc="sum", fill_value=0, margins=True, margins_name="Total")
print("(b) Weekly transaction value by county: all clean transactions")
display(weekly_pivot)

weekly_ex = weekly[~suspect]
weekly_pivot_ex = weekly_ex.pivot_table(index="week_starting", columns="county", values="amount",
                                        aggfunc="sum", fill_value=0, margins=True, margins_name="Total")
print("(b2) Weekly transaction value by county: quarantined extreme values excluded")
display(weekly_pivot_ex)
assert weekly_pivot.loc["Total", "Total"] == clean["amount"].sum()
print("Days of data in first/last week:", weekly.groupby("week_starting")["txn_date"].nunique().iloc[[0, -1]].to_dict())

(b) Weekly transaction value by county: all clean transactions


county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown,Total
week_starting,,,,,,,
2026-02-23,0.0,1893.0,6759.0,7046.0,2684.0,0.0,18382.0
2026-03-02,18317.0,287933.0,285610.0,68967.0,56574.0,0.0,717401.0
2026-03-09,267641.0,23186.0,30418.0,52801.0,28730.0,0.0,402776.0
2026-03-16,22054.0,27935.0,25085.0,43048.0,39795.0,1639.0,159556.0
2026-03-23,11504.0,27077.0,17050.0,41314.0,18868.0,0.0,115813.0
2026-03-30,22609.0,41077.0,38344.0,282931.0,18145.0,8507.0,411613.0
2026-04-06,15030.0,15341.0,47033.0,56727.0,276609.0,9824.0,420564.0
2026-04-13,12265.0,29297.0,26730.0,56302.0,26093.0,0.0,150687.0
2026-04-20,24507.0,37439.0,16687.0,64926.0,30862.0,1877.0,176298.0


(b2) Weekly transaction value by county: quarantined extreme values excluded


county,Kiambu,Kisumu,Mombasa,Nairobi,Nakuru,Unknown,Total
week_starting,,,,,,,
2026-02-23,0.0,1893.0,6759.0,7046.0,2684.0,0.0,18382.0
2026-03-02,18317.0,37933.0,35610.0,68967.0,56574.0,0.0,217401.0
2026-03-09,17641.0,23186.0,30418.0,52801.0,28730.0,0.0,152776.0
2026-03-16,22054.0,27935.0,25085.0,43048.0,39795.0,1639.0,159556.0
2026-03-23,11504.0,27077.0,17050.0,41314.0,18868.0,0.0,115813.0
2026-03-30,22609.0,41077.0,38344.0,32931.0,18145.0,8507.0,161613.0
2026-04-06,15030.0,15341.0,47033.0,56727.0,26609.0,9824.0,170564.0
2026-04-13,12265.0,29297.0,26730.0,56302.0,26093.0,0.0,150687.0
2026-04-20,24507.0,37439.0,16687.0,64926.0,30862.0,1877.0,176298.0


Days of data in first/last week: {'2026-02-23': 1, '2026-04-27': 3}


### What the tables say
- **Nairobi leads on every measure:** 180 transactions (30.0% of all) and KES 704,866 in value (all clean data), against 133 transactions and KES 517,340 for Mombasa. Excluding the quarantined values the top county is still Nairobi (KES 454,866), and the order of the five counties by value does not change (Nairobi, Mombasa, Nakuru, Kisumu, Kiambu), so the ranking is robust to that decision. Nakuru and Kisumu are effectively tied on value (KES 513,235 vs KES 512,486, 0.1% apart), but Kisumu handles clearly more transactions (111 vs 99), so it handles more traffic at a lower average value.
- **Deposits dominate by count** (373 of 600 transactions, 62%), and by value excluding the quarantined values (KES 923,129 deposits vs KES 513,611 withdrawals). Including them, the 5 extreme values (3 deposits, 2 withdrawals) inflate the totals to KES 1,673,129 and KES 1,013,611.
- **Average transaction values are misleading unless quarantined:** the overall average falls from KES 4,478 to KES 2,415 when the five extreme values are excluded; in a county with about 100 transactions, a single KES 250,000 value adds roughly KES 2,500 to that county's average on its own.
- **Top agents (value, quarantined values excluded):** AG006 (Nakuru, KES 109,958), AG007 (Nairobi, KES 106,573), AG017 (Nakuru, KES 98,796). Only the register-listed agents are ranked; the 7 unknown-agent transactions cannot be attributed.
- **Weekly pattern:** the busiest full week by value is the week starting 2026-03-02 (KES 217,401, excluding quarantined values); the first week (1 March only) and the last (27–29 April) are partial, so they are not comparable. In the all-data pivot, the weeks that contain an extreme value (2026-03-02, 2026-03-09, 2026-03-30, 2026-04-06) total KES 402,776–KES 717,401, against KES 115,813–KES 176,298 for the other full weeks: spikes created by five transactions, which is another reason to verify them before reporting weekly trends.

## 8. Export the clean file and verify its contract

In [16]:
Path("outputs").mkdir(exist_ok=True)
out_cols = ["txn_id", "agent_id", "txn_type", "amount", "txn_date", "amount_imputed", "is_outlier", "county", "unknown_agent", "onboarded"]
clean[out_cols].to_csv("outputs/agent_transactions_clean.csv", index=False, date_format="%Y-%m-%d")

out = pd.read_csv("outputs/agent_transactions_clean.csv")
assert out["txn_id"].is_unique
assert set(out["txn_type"]) == {"Deposit", "Withdrawal"}
assert pd.api.types.is_numeric_dtype(out["amount"]) and out["amount"].notna().all()
assert out["txn_date"].str.fullmatch(r"\d{4}-\d{2}-\d{2}").all()
assert {"amount_imputed", "is_outlier", "county"} <= set(out.columns)
assert len(out) == len(clean)
print("outputs/agent_transactions_clean.csv:", out.shape)
print("Row count BEFORE cleaning:", len(txns_raw), "| AFTER cleaning:", len(out), "| removed:", len(txns_raw) - len(out))
out.head()

outputs/agent_transactions_clean.csv: (600, 10)
Row count BEFORE cleaning: 612 | AFTER cleaning: 600 | removed: 12


,txn_id,agent_id,txn_type,amount,txn_date,amount_imputed,is_outlier,county,unknown_agent,onboarded
0,TX00001,AG009,Deposit,353.0,2026-03-03,False,False,Nakuru,False,2024-09-01
1,TX00002,AG020,Deposit,1238.0,2026-04-27,False,False,Mombasa,False,2025-08-01
2,TX00003,AG008,Deposit,9129.0,2026-04-29,False,True,Kisumu,False,2024-08-01
3,TX00004,AG009,Deposit,652.0,2026-04-24,False,False,Nakuru,False,2024-09-01
4,TX00005,AG009,Withdrawal,478.0,2026-04-17,False,False,Nakuru,False,2024-09-01


---

# Data Quality Report: agent transaction extract

**Scope:** `agent_transactions.csv` (612 rows) joined to `agents.csv` (20 agents). **Result:** 612 rows → **600 clean rows** (12 duplicates removed), with every other issue flagged or corrected and logged by `clean_agent_data()`.

| Dimension | Finding (raw rows) | Treatment | Residual risk |
|---|---|---|---|
| **Completeness** | 15 blank + 7 `"N/A"` amounts (= 22 under `isna()`; 20 transactions after de-duplication, 3.3%) | Flagged `amount_imputed`, then filled with the same-type median of genuine non-outlier amounts (Deposit KES 1,717, Withdrawal KES 1,646); not dropped | Imputed values add 1.3% of value; do not use for agent-level judgements |
| **Validity** | 5 dates as `YYYY/MM/DD`; `txn_date`, `onboarded` held as text; amounts contain the text `"N/A"` | Normalised and converted to `datetime` / numeric; 0 unparseable dates remain | None known |
| **Accuracy** | 40 IQR outliers: 35 plausible tail values and 5 identical values of KES 250,000 (46.5% of value) | All flagged `is_outlier`, none deleted; the 5 extremes are quarantined from averages and shown with/without totals | **High:** the 5 extremes must be verified against the source ledger |
| **Consistency** | 384 `txn_type` rows in mixed case/spacing (111 with trailing spaces); 8 rows with agent `AG099` not in the register | Standardised to exactly "Deposit" / "Withdrawal"; left join validated many-to-one; 7 unknown-agent transactions kept as `county="Unknown"`, `unknown_agent=True` | Cannot attribute KES 27,156 (1.0% of value) to a county |
| **Uniqueness** | 12 duplicate transactions; only 11 were exact copies (one copy had `agent_id` overwritten) | De-duplicated on `txn_id`, keeping the copy with a registered agent | None known |
| **Timeliness** | Transactions run 01 Mar to 29 Apr 2026; 0 dated in the future or before the agent's onboarding | None needed | First and last weeks are partial; one snapshot, so no trend beyond ~8 weeks |

**Questions for the data owner:** (1) is KES 250,000 a placeholder, a system limit or a real amount? (2) what is `AG099`? (3) can the 20 missing amounts be recovered from the source ledger?

**Privacy and the Kenya Data Protection Act 2019.** This extract holds no names or phone numbers, but if it also contained **customer phone numbers**, that field would be direct personal data, and so would every transaction row linked to it (`txn_id`, `amount`, `txn_date`, `txn_type` and `agent_id` then describe a named person's financial activity). `agent_id` and `county` may also identify an individual agent, and a county + date + unusual amount can single someone out. I would: **(1) minimise** by not loading the phone column unless the analysis needs it; **(2) pseudonymise** it with a keyed, salted hash (e.g. HMAC-SHA256, key stored outside the repository) so customers stay countable but not identifiable, and mask it (`0712 *** 678`) for display; **(3) aggregate** outputs and suppress any group of fewer than about 5 customers; **(4) secure** the data (restricted access, encryption at rest and in transit) and keep raw files out of GitHub via `.gitignore`; and **(5) limit purpose and retention**, with a documented lawful basis, customer notice, and a data-protection impact assessment, because this is large-scale financial data; the provider should be registered with the Office of the Data Protection Commissioner and must not move the data outside Kenya without the safeguards the Act requires.